In [ ]:
# notebook for filtering for the promoter emVars for the genes Steve asked about

In [5]:
# import packages
import pandas as pd
from tqdm import tqdm
import os

In [6]:
# define function to convert predictions to final df
def vcf2df (pred_df):
    # make lists of preds for new column
    # k562
    k_ref = []
    k_alt = []
    k_skew = []
    # hepg2
    h_ref = []
    h_alt = []
    h_skew = []
    # sknsh
    s_ref = []
    s_alt = []
    s_skew = []
    gene_id = []
    # parse predictions in 'INFO' column
    for i in tqdm(pred_df[7]):
        all_preds = i.split(';')
        # k562
        k_ref.append(float(all_preds[0].split('=')[-1]))
        k_alt.append(float(all_preds[3].split('=')[-1]))
        k_skew.append(float(all_preds[6].split('=')[-1]))
        # hepg2
        h_ref.append(float(all_preds[1].split('=')[-1]))
        h_alt.append(float(all_preds[4].split('=')[-1]))
        h_skew.append(float(all_preds[7].split('=')[-1]))
        # sknsh
        s_ref.append(float(all_preds[2].split('=')[-1]))
        s_alt.append(float(all_preds[5].split('=')[-1]))
        s_skew.append(float(all_preds[8].split('=')[-1]))
        # add annotated gene name
        #gene_id.append(all_preds[-1].split('=')[-1])
    
    df = pd.DataFrame({'chrom' : pred_df[0],
                       'pos' : pred_df[1],
                       'id' : pred_df[2],
                       'ref' : pred_df[3],
                       'alt' : pred_df[4],
                       'k562_ref_pred' : k_ref,
                       'k562_alt_pred' : k_alt,
                       'k562_skew_pred' : k_skew,
                       'hepg2_ref_pred' : h_ref,
                       'hepg2_alt_pred' : h_alt,
                       'hepg2_skew_pred' : h_skew,
                       'sknsh_ref_pred' : s_ref,
                       'sknsh_alt_pred' : s_alt,
                       'sknsh_skew_pred' : s_skew,
                       'gene_id' : [i.split('..')[0].split('_')[0] for i in pred_df[2]],
                       'transcript_id' : [i.split('..')[0].split('_')[1] for i in pred_df[2]],
                       'hgnc_id' : [i.split('..')[0].split('_')[-1] for i in pred_df[2]]})
    return df

In [10]:
# open the grep filtered predictions and concatenate
# all were filterered as follows:
# gzip -cd all.gencode.v44.canonical.protein.coding.1kb.promoters.sat.mut.updated.pos.sorted.vcf.gz | grep GENE_NAME > cancerGenes4steve/genename.1kb.promoter.sat.mut.preds.vcf
preds2reformat = pd.concat([pd.read_csv(f'../mpac/processed_data/cancerGenes4steve/{i}', sep = '\t', header = None) for i in os.listdir('../mpac/processed_data/cancerGenes4steve/')])

In [26]:
# open tabix filtered (tabix_filter_mpac_preds.sh) predictions and reformat for downstream analyses
mpac_preds = vcf2df(preds2reformat)
# add emvar status
# cell types
mpac_preds.loc[:, 'k562_emvar'] = [1 if i > 0.5 else 0 for i in mpac_preds['k562_skew_pred'].abs()]
mpac_preds.loc[:, 'hepg2_emvar'] = [1 if i > 0.5 else 0 for i in mpac_preds['hepg2_skew_pred'].abs()]
mpac_preds.loc[:, 'sknsh_emvar'] = [1 if i > 0.5 else 0 for i in mpac_preds['sknsh_skew_pred'].abs()]
# any
mpac_preds.loc[:, 'emvar_any'] = [1 if sum([k,h,s]) > 0 else 0 for k,h,s in zip(mpac_preds['k562_emvar'], mpac_preds['hepg2_emvar'], mpac_preds['sknsh_emvar'])]

100%|██████████| 15000/15000 [00:00<00:00, 281784.37it/s]


In [28]:
# save to disk
# all
mpac_preds.to_csv('../mpac/processed_data/cancerGenes4steve/MLH1.MSH2.MSH6.EPCAM.PMS2.1kb.sat.mut.all.tsv', sep = '\t', index = False)
# mavars
mpac_preds[mpac_preds['emvar_any'] == 1].to_csv('../mpac/processed_data/cancerGenes4steve/MLH1.MSH2.MSH6.EPCAM.PMS2.1kb.sat.mut.emVars.tsv', sep = '\t', index = False)